In [0]:
"""
Set up Spark access to Azure Data Lake Storage (ADLS) 
using the account access key and list all files in the lakehouse container 
to verify the connection.
"""

spark.conf.set(
    "fs.azure.account.key.projectlinkedinjobs.dfs.core.windows.net",
    "<AccessKey>"
)

display(
    dbutils.fs.ls("abfss://lakehouse@projectlinkedinjobs.dfs.core.windows.net/")
)


path,name,size,modificationTime
abfss://lakehouse@projectlinkedinjobs.dfs.core.windows.net/gold/,gold/,0,1763212873000
abfss://lakehouse@projectlinkedinjobs.dfs.core.windows.net/processed/,processed/,0,1763101513000
abfss://lakehouse@projectlinkedinjobs.dfs.core.windows.net/raw/,raw/,0,1763056713000


In [0]:
"""
Install all required Python libraries for text processing and feature engineering. 
Includes NLTK for sentiment analysis, Sentence Transformers for embeddings, 
TextStat for readability metrics, and TextBlob for subjectivity analysis.
"""

%pip install nltk
%pip install sentence-transformers
%pip install textstat
%pip install textblob

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/176.4 kB ? eta -:--:--
   ━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/176.4 kB ? eta -:--:--
   ━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/176.4 kB 967.5 kB/s eta 0:00:01
   ━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/176.4 kB 367.6 kB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━ 92.2/176.4 kB 688.5 kB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━ 92.2/176.4 kB 688.5 kB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━ 133.1/176.4 kB 645.1 kB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.4/176.4 kB 766.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/2.1 MB ? eta -:--:--
   ━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.1/2.1 MB 4

In [0]:
"""
Import all required libraries for Spark, text processing, and feature engineering. 
Includes Spark SQL and ML components for data manipulation and TF-IDF, 
NLTK, TextBlob, TextStat, and Sentence Transformers for NLP tasks, 
and NumPy, Pandas, and scikit-learn for data analysis and vectorization.
"""

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, year, month, dayofweek, to_timestamp,
    length, size, split, udf
)

from pyspark.sql.types import (
    StringType, DoubleType, FloatType, ArrayType
)
from pyspark.ml.feature import (
    Tokenizer, StopWordsRemover, CountVectorizer, IDF
)
from pyspark.ml import Pipeline

import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from textblob import TextBlob
import textstat
from sentence_transformers import SentenceTransformer

import numpy as np
import pandas as pd
import re

from sklearn.feature_extraction.text import TfidfVectorizer


2025-11-19 18:05:47.418711: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-19 18:05:47.422119: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-19 18:05:47.430755: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-11-19 18:05:47.446406: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-11-19 18:05:47.451058: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-11-19 18:05:47.465061: I tensorflow/core/platform/cpu_feature_gu

In [0]:
"""
Load the cleaned dataset from the Gold layer (features_v1) stored in Delta format. 
Display the schema to verify column types and show a sample of the first ten records.
"""

df = spark.read.format("delta").load(
    "abfss://lakehouse@projectlinkedinjobs.dfs.core.windows.net/gold/linkedin_features_v1/"
)

df.printSchema()
display(df.limit(10))

root
 |-- company: string (nullable = true)
 |-- search_country: string (nullable = true)
 |-- job_level: string (nullable = true)
 |-- job_type: string (nullable = true)
 |-- job_skills: string (nullable = true)
 |-- job_summary: string (nullable = true)
 |-- job_role: string (nullable = true)
 |-- label: double (nullable = true)



company search_country job_level job_type job_skills job_summary job_role label bose corporation united states associate hybrid calendar support, travel support, sap delegate responsibilities, invoice approval and reconciliation, purchase requisition, time administration, event coordination, administrative support, ms office suite (excel powerpoint outlook), sap, ariba, sharepoint job description you know the moment. it’s the first notes of that song you love, the intro to your favorite movie, or simply the sound of someone you love saying “hello.” it’s in these moments that sound matters most. at bose, we believe sound is the most powerful force on earth. we’ve dedicated ourselves to improving it for nearly 60 years. and we’re passionate down to our bones about making whatever you’re listening to a little more magical. about the marketing team the marketing team at bose consists of passionate, bold, and music,oving storytellers. we tap into the magic of what makes bose, bose, and through our marketing efforts, connect that magic with people who relate to our belief that sound is the most powerful force on earth. about the role we have a great opportunity for an upbeat, organized, and creative sr. administrative assistant to support two vp's within our marketing team. this role will be an opportunity for someone who is looking to broaden their knowledge of marketing. this role will work out of our framingham, ma hqs in a hybrid capacity; tues,ed,hursday in office. what you will do: as a member of the marketing team, you will provide calendar support to marketing vps and their broader teams, assist with travel support, submitting travel expenses and sap delegate responsibilities including approving and reconciling invoices when needed. the task complexity will vary based on team initiatives and functional needs of these teams. you will be expected to provide administrative support to multiple directors and senior level managers, which includes coordinating calendar meetings, travel and preparing expense reports and providing preparation for internal and external meetings. you will handle purchase requisitions for products and services across the group, administrate time and coordinate events as the need arises. you will work with other administrative,xecutive assistants as needed. what you need for this job: prior experience supporting multiple executives in a fast,aced environment. the successful candidate is an outstanding organizer, who can multitask and prioritize optimally in a fast,aced environment. problem solvers with strong initiative and good judgment will be highly valued. customer service focused individuals, with strong communication skills who can build,anage relationships and work collaboratively will be a phenomenal teammate who is resourceful, self,otivated and carry a "can do" positive demeanor are desired. lastly, strong pc application skills, e.g., ms office suite, particularly excel, powerpoint, and outlook, are a must. sap, ariba, sharepoint, experience encouraged. show more show less administrative assistant 65.0 the ohio state university wexner medical center united states mid senior onsite computers, customer service, microsoft office, medical terminology, records management, scheduling, event planning, research, data entry, powerpoint, microsoft word, email, cv writing, cme documentation, background check, drug screen, physical exam screen reader users may encounter difficulty with this site. for assistance with applying, please contact hr,ccessibleapplication@osu.edu. if you have questions while submitting an application, please review these frequently asked questions. current employees and students if you are currently employed or enrolled as a student at the ohio state university, please log in to workday to use the internal application process. welcome to the ohio state university's career site. we invite you to apply to positions of interest. in order to ensure your application is complete, you mu

In [0]:
"""
Split the dataset into training, validation, and test subsets using a 70/15/15 ratio. 
Count the number of samples in each split and calculate the percentage distribution 
to confirm the split proportions.
"""

train, val, test = df.randomSplit([0.7, 0.15, 0.15], seed=42)

train_count = train.count()
val_count = val.count()
test_count = test.count()

total = train_count + val_count + test_count

train_pct = (train_count / total) * 100
val_pct = (val_count / total) * 100
test_pct = (test_count / total) * 100

print(f"Train: {train_pct:.2f}% | Validation: {val_pct:.2f}% | Test: {test_pct:.2f}%")


Train: 70.09% | Validation: 14.96% | Test: 14.95%


In [0]:
"""
Save the train, validation, and test DataFrames to the Gold layer in Delta format. 
Each split is stored in a separate folder for consistent access and reuse 
during model training and evaluation.
"""

train.write.format("delta").mode("overwrite").save(
    "abfss://lakehouse@projectlinkedinjobs.dfs.core.windows.net/gold/feature_v2/train/"
)

val.write.format("delta").mode("overwrite").save(
    "abfss://lakehouse@projectlinkedinjobs.dfs.core.windows.net/gold/feature_v2/validation/"
)

test.write.format("delta").mode("overwrite").save(
    "abfss://lakehouse@projectlinkedinjobs.dfs.core.windows.net/gold/feature_v2/test/"
)


In [0]:
"""
Load the training dataset from the Gold layer (feature_v2/train) stored in Delta format. 
Display the first ten records to verify successful loading and data integrity.
"""

train = spark.read.format("delta").load(
    "abfss://lakehouse@projectlinkedinjobs.dfs.core.windows.net/gold/feature_v2/train/"
)

display(train.limit(10))


company search_country job_level job_type job_skills job_summary job_role label #twiceasnice recruiting united states mid senior onsite customer service, erp, crm, order processing, billing, account management, communication, problem solving, teamwork, flexibility, adaptability, proactivity, attention to detail, organizational skills, time management, microsoft office suite no_information customer service representative 0.0 1199seiu benefit and pension funds united states mid senior onsite project management, microsoft office suite, visio, key performance indicators, tafthartley fund, budgeting, variance reports, financial documents, contract execution, data analysis, presentation skills, meeting agenda preparation, multitasking, prioritization, high school diploma, bachelor's degree, administrative assistant experience, executive assistant experience, communication skills, organization skills, work ethic, selfmotivation responsibilities assist the chief of benefit operations and the senior director of benefit operations with documenting and tracking projects and follow up on project tasks collaborate with departmental heads to document and track electronically the progress of plans including plan of work and corrective action plans, complaints, inquiries, and appeals implement,aintain benefit operations meetings, and track,onitor key discussion points by formulating task list for attendees and chief,enior director of benefit operations monitor production of monthly dashboard and executive summary report to identify and report trends for discussion coordinate related activities to ensure timely review or preparation of standing, or ad hoc request that impact all benefit operations areas such as budget preparation, variance reports, financial,ayment documents, contract execution, and document reviews collaborate with internal and external personnel to compile, organize, analyze, and present information so it can be useful in making decisions prepare meeting agendas and compile documents in advance handle several projects in parallel and follow through on expectations, with minimal supervision perform special projects and assignments as directed by management qualifications high school diploma or ged required; bachelor’s degree preferred minimum of four (4) years administrative assistant experience supporting senior level management and two (2) years of work experience in a health plan,ealth related organization required proven experience as an executive administrative assistant, senior executive assistant intermediate computer skills: microsoft word, excel, powerpoint, and visio required experience with preparation and development of key performance indicators ability to multitask and prioritize daily workload project management experience preferred knowledge of taft,artley fund (non,rofit sector) preferred detailed,riented and possess strong communication and organization skills strong work ethic, highly self,otivated, and understand the need we offer extraordinary benefits including outstanding health, dental, pension and family benefits for most positions which are paid entirely by the funds without co,ayments, deductibles, or out,f,ocket expenses for covered services. we also offer tuition reimbursement, generous holiday, vacation, and sick leave, as well as a 401k plan. show more show less administrative assistant 65.0 1st employment united states mid senior onsite netsuite, financial audits, master's degree in finance or accounting, historical finance experience, financial reporting, financial forecasting, financial review, analytical skills, problemsolving skills, written and verbal communication abilities, detailoriented, collaborative work, multiple task management, deadline management location: fayetteville, ar salary range: $70,000 – $95,000 per year job description: our client, a leading company in fayetteville, arkansas, is seeking a highly skilled and experienced senior financial analyst to join their finance team

In [0]:
from pyspark.sql.functions import col, concat_ws

train = train.withColumn(
    "raw_text",
    concat_ws(" ", col("job_summary"), col("job_skills"))
)


In [0]:
import re
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import size, split, length, col

@F.udf(returnType=StringType())
def clean_text(text):
    if not text:
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " url ", text)   # replace URLs
    text = re.sub(r"\d+", " num ", text)              # replace numbers
    text = re.sub(r"[^a-z\s]", "", text)              # keep only letters + spaces
    text = re.sub(r"\s+", " ", text).strip()          # collapse spaces
    return text


In [0]:
# clean combined text
clean_train = train.withColumn("clean_text", clean_text(F.col("raw_text")))

# drop very short descriptions (same idea as lab: length > 10 chars)
clean_train = clean_train.filter(F.length("clean_text") > 10)

# add word and char length features
clean_train = (
    clean_train
    .withColumn("text_length_words", size(split(col("clean_text"), " ")))
    .withColumn("text_length_chars", length(col("clean_text")))
)

display(clean_train.select("raw_text", "clean_text", "text_length_words", "text_length_chars").limit(10))


raw_text clean_text text_length_words text_length_chars description ability to supervise any area of the distribution center or production reviews work load requirements and staffs shift ensuring adequate coverage and within budgeted guidelines. ensures productivity and cost controls are met or exceeded. creates a climate providing motivation, participation and opportunities for employees to do their best. prepares staff for future opportunities and takes an active role in the development of future leaders. impacts retention by fostering a respectful team environment that supports workplace values through professional communication and timely problem solving. responsible for seasonal planning by developing continuous improvement programs and procedures. responsible for developing training materials and procedures for department. ensures all department procedures (sop) are documented and utlized. ensures system training and appropriate documentation to ensure systems are utilized and users are systems saavy (sap) collaborate with peer supervisors to promote cross,epartmental cooperation and ensure clear communication monitor employee performance and create and deliver pip's and pdr's for direct and indirect reports. assists in interviewing and assessing leadership candidates as assigned. communicates effectively both horizontally and vertically maintaining two,ay communications responsible for department and company safety and security programs, safety audits and ensuring the safe conduct of all associates within their assigned area. coordinate with qa to ensure sqf and fsma requirements are met assist in maintaining the security of the facility requirements education , bachelor's degree preferred. high school diploma required. the top candidate for this position will have three to five years of leadership experience in a similar role or level of responsibility. time management, organizational skills and the ability to influence and motivate a team previous warehouse management system and sap knowledge desired. ability to analyze reports and draw conclusions,recommendations knowledge of microsoft office effective written and verbal communication skills show more show less supervision, workload management, cost control, motivation, leadership development, team building, continuous improvement, training and development, documentation, sap, crossdepartmental collaboration, performance management, interviewing and assessment, twoway communication, safety and security management, sqf and fsma compliance, facility security, high school diploma, bachelor's degree (preferred), leadership experience, time management, organizational skills, influencing and motivating skills, warehouse management system, microsoft office, data analysis, written and verbal communication description ability to supervise any area of the distribution center or production reviews work load requirements and staffs shift ensuring adequate coverage and within budgeted guidelines ensures productivity and cost controls are met or exceeded creates a climate providing motivation participation and opportunities for employees to do their best prepares staff for future opportunities and takes an active role in the development of future leaders impacts retention by fostering a respectful team environment that supports workplace values through professional communication and timely problem solving responsible for seasonal planning by developing continuous improvement programs and procedures responsible for developing training materials and procedures for department ensures all department procedures sop are documented and utlized ensures system training and appropriate documentation to ensure systems are utilized and users are systems saavy sap collaborate with peer supervisors to promote crossepartmental cooperation and ensure clear communication monitor employee performance and create and deliver pips and pdrs for direct and indirect reports assists in interviewing and as

In [0]:
val  = val.withColumn("raw_text",  concat_ws(" ", col("job_summary"), col("job_skills")))
test = test.withColumn("raw_text", concat_ws(" ", col("job_summary"), col("job_skills")))

clean_val = (
    val.withColumn("clean_text", clean_text(F.col("raw_text")))
       .filter(F.length("clean_text") > 10)
       .withColumn("text_length_words", size(split(col("clean_text"), " ")))
       .withColumn("text_length_chars", length(col("clean_text")))
)

clean_test = (
    test.withColumn("clean_text", clean_text(F.col("raw_text")))
        .filter(F.length("clean_text") > 10)
        .withColumn("text_length_words", size(split(col("clean_text"), " ")))
        .withColumn("text_length_chars", length(col("clean_text")))
)


In [0]:
"""
Compute sentiment scores on cleaned job text (summary + skills).
For each record, calculate positive, neutral, negative, and compound
sentiment values using NLTK VADER. Then save sentiment-enhanced
train/val/test datasets to the Gold layer.
"""

import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from pyspark.sql.types import DoubleType
import pyspark.sql.functions as F

# We already have: clean_train, clean_val, clean_test
# (from your previous cleaning block)

# 1) Setup VADER
nltk.download('vader_lexicon')
analyzer = SentimentIntensityAnalyzer()

def make_sent_udf(kind):
    def _f(text):
        if not text:
            return 0.0
        scores = analyzer.polarity_scores(text)
        return float(scores[kind])
    return F.udf(_f, DoubleType())

sent_pos_udf      = make_sent_udf("pos")
sent_neu_udf      = make_sent_udf("neu")
sent_neg_udf      = make_sent_udf("neg")
sent_comp_udf     = make_sent_udf("compound")

# 2) Apply to TRAIN
sentiment_train = (
    clean_train
    .withColumn("sentiment_pos",      sent_pos_udf("clean_text"))
    .withColumn("sentiment_neu",      sent_neu_udf("clean_text"))
    .withColumn("sentiment_neg",      sent_neg_udf("clean_text"))
    .withColumn("sentiment_compound", sent_comp_udf("clean_text"))
)

display(
    sentiment_train.select(
        "clean_text",
        "sentiment_pos",
        "sentiment_neu",
        "sentiment_neg",
        "sentiment_compound"
    ).limit(10)
)

# 3) Apply to VAL and TEST as well
sentiment_val = (
    clean_val
    .withColumn("sentiment_pos",      sent_pos_udf("clean_text"))
    .withColumn("sentiment_neu",      sent_neu_udf("clean_text"))
    .withColumn("sentiment_neg",      sent_neg_udf("clean_text"))
    .withColumn("sentiment_compound", sent_comp_udf("clean_text"))
)

sentiment_test = (
    clean_test
    .withColumn("sentiment_pos",      sent_pos_udf("clean_text"))
    .withColumn("sentiment_neu",      sent_neu_udf("clean_text"))
    .withColumn("sentiment_neg",      sent_neg_udf("clean_text"))
    .withColumn("sentiment_compound", sent_comp_udf("clean_text"))
)


base_path = "abfss://lakehouse@projectlinkedinjobs.dfs.core.windows.net/gold/linkedin_feature_v2/"

sentiment_train.write.format("delta").mode("overwrite").save(base_path + "sentiment_train/")
sentiment_val.write.format("delta").mode("overwrite").save(base_path + "sentiment_val/")
sentiment_test.write.format("delta").mode("overwrite").save(base_path + "sentiment_test/")


[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


clean_text sentiment_pos sentiment_neu sentiment_neg sentiment_compound description ability to supervise any area of the distribution center or production reviews work load requirements and staffs shift ensuring adequate coverage and within budgeted guidelines ensures productivity and cost controls are met or exceeded creates a climate providing motivation participation and opportunities for employees to do their best prepares staff for future opportunities and takes an active role in the development of future leaders impacts retention by fostering a respectful team environment that supports workplace values through professional communication and timely problem solving responsible for seasonal planning by developing continuous improvement programs and procedures responsible for developing training materials and procedures for department ensures all department procedures sop are documented and utlized ensures system training and appropriate documentation to ensure systems are utilized and users are systems saavy sap collaborate with peer supervisors to promote crossepartmental cooperation and ensure clear communication monitor employee performance and create and deliver pips and pdrs for direct and indirect reports assists in interviewing and assessing leadership candidates as assigned communicates effectively both horizontally and vertically maintaining twoay communications responsible for department and company safety and security programs safety audits and ensuring the safe conduct of all associates within their assigned area coordinate with qa to ensure sqf and fsma requirements are met assist in maintaining the security of the facility requirements education bachelors degree preferred high school diploma required the top candidate for this position will have three to five years of leadership experience in a similar role or level of responsibility time management organizational skills and the ability to influence and motivate a team previous warehouse management system and sap knowledge desired ability to analyze reports and draw conclusionsrecommendations knowledge of microsoft office effective written and verbal communication skills show more show less supervision workload management cost control motivation leadership development team building continuous improvement training and development documentation sap crossdepartmental collaboration performance management interviewing and assessment twoway communication safety and security management sqf and fsma compliance facility security high school diploma bachelors degree preferred leadership experience time management organizational skills influencing and motivating skills warehouse management system microsoft office data analysis written and verbal communication 0.257 0.736 0.006 0.9981 job summary the primary responsibility of the business development manager is to prospect grow and nurture new business opportunities within the pittsburgh metro area with a focus on generating new revenue and improving profitability in accordance with num krew business objectives this position is tasked with prospecting to identify new potential opportunities and progress them through the sales cycle converting leads into customers the bdm develops and maintains positive relationships with new and existing commercial customers by highlighting industry knowledge and expertise backed by the companys reputation this position supports a fastaced growth environment and regularly communicates with the internal sales and leadership teams this bdm is a salarylusommission position there will also be bonus opportunities based on the overall achievement of the position kpis both the commission scale and kpionus structure are available upon request duties and responsibilities leverages various contact methods such as cold calling or inerson meetings to secure profitable sales through contact lists and selfenerated leads to grow yoy results accomplishes business development by researching and developing

In [0]:
"""
Extract TF-IDF features from cleaned job text (summary + skills).
We use a Spark ML pipeline with:
- Tokenizer
- StopWordsRemover
- CountVectorizer (TF)
- IDF (TF-IDF)

We fit the pipeline on TRAIN only, then transform train/val/test
and save the TF-IDF feature datasets to the Gold layer.
"""

from pyspark.ml.feature import Tokenizer, StopWordsRemover, CountVectorizer, IDF
from pyspark.ml import Pipeline

# 1) Define the text feature pipeline on `clean_text`
tokenizer = Tokenizer(inputCol="clean_text", outputCol="words")
remover   = StopWordsRemover(inputCol="words", outputCol="filtered_words")

cv = CountVectorizer(
    inputCol="filtered_words",
    outputCol="raw_features",
    vocabSize=5000  # you can increase if you want (e.g. 20000)
)

idf = IDF(
    inputCol="raw_features",
    outputCol="tfidf_features"
)

pipeline = Pipeline(stages=[tokenizer, remover, cv, idf])

# 2) Fit on TRAIN only (important!)
tfidf_model = pipeline.fit(sentiment_train)

# 3) Transform TRAIN / VAL / TEST
tfidf_train = tfidf_model.transform(sentiment_train)
tfidf_val   = tfidf_model.transform(sentiment_val)
tfidf_test  = tfidf_model.transform(sentiment_test)

tfidf_train.cache()

display(
    tfidf_train.select("clean_text", "tfidf_features").limit(10)
)

# 4) Save to your Gold layer (adjust container + account)
base_path = "abfss://lakehouse@projectlinkedinjobs.dfs.core.windows.net/gold/linkedin_feature_v2/"

tfidf_train.write.format("delta").mode("overwrite").save(base_path + "tfidf_train/")
tfidf_val.write.format("delta").mode("overwrite").save(base_path + "tfidf_val/")
tfidf_test.write.format("delta").mode("overwrite").save(base_path + "tfidf_test/")


Uploading artifacts:   0%|          | 0/4 [00:00<?, ?it/s]

clean_text,tfidf_features
description ability to supervise any area of the distribution center or production reviews work load requirements and staffs shift ensuring adequate coverage and within budgeted guidelines ensures productivity and cost controls are met or exceeded creates a climate providing motivation participation and opportunities for employees to do their best prepares staff for future opportunities and takes an active role in the development of future leaders impacts retention by fostering a respectful team environment that supports workplace values through professional communication and timely problem solving responsible for seasonal planning by developing continuous improvement programs and procedures responsible for developing training materials and procedures for department ensures all department procedures sop are documented and utlized ensures system training and appropriate documentation to ensure systems are utilized and users are systems saavy sap collaborate with peer supervisors to promote crossepartmental cooperation and ensure clear communication monitor employee performance and create and deliver pips and pdrs for direct and indirect reports assists in interviewing and assessing leadership candidates as assigned communicates effectively both horizontally and vertically maintaining twoay communications responsible for department and company safety and security programs safety audits and ensuring the safe conduct of all associates within their assigned area coordinate with qa to ensure sqf and fsma requirements are met assist in maintaining the security of the facility requirements education bachelors degree preferred high school diploma required the top candidate for this position will have three to five years of leadership experience in a similar role or level of responsibility time management organizational skills and the ability to influence and motivate a team previous warehouse management system and sap knowledge desired ability to analyze reports and draw conclusionsrecommendations knowledge of microsoft office effective written and verbal communication skills show more show less supervision workload management cost control motivation leadership development team building continuous improvement training and development documentation sap crossdepartmental collaboration performance management interviewing and assessment twoway communication safety and security management sqf and fsma compliance facility security high school diploma bachelors degree preferred leadership experience time management organizational skills influencing and motivating skills warehouse management system microsoft office data analysis written and verbal communication,"Map(vectorType -> sparse, length -> 5000, indices -> List(1, 2, 3, 4, 5, 6, 9, 10, 12, 15, 17, 18, 22, 28, 29, 30, 31, 32, 34, 35, 39, 41, 44, 45, 49, 50, 52, 55, 59, 74, 76, 77, 80, 82, 88, 89, 94, 97, 100, 101, 111, 114, 116, 120, 123, 126, 136, 139, 146, 148, 149, 151, 155, 158, 165, 167, 168, 169, 172, 173, 180, 191, 202, 211, 237, 245, 248, 251, 253, 254, 260, 280, 284, 286, 292, 300, 306, 318, 331, 333, 335, 339, 340, 351, 353, 359, 370, 388, 395, 435, 450, 454, 456, 477, 490, 501, 529, 539, 555, 568, 585, 601, 605, 632, 653, 697, 706, 729, 753, 767, 772, 809, 825, 850, 861, 871, 883, 890, 891, 907, 960, 1049, 1103, 1133, 1138, 1167, 1188, 1190, 1228, 1229, 1348, 1357, 1456, 1461, 1496, 1730, 1742, 1841, 2250, 2268, 2377, 2440, 2462, 2783, 2870, 2978, 3034, 3119, 3122, 3265, 3541, 3668, 3780, 3839, 4231, 4464, 4636), values -> List(0.9066329921977608, 3.6521281900695506, 0.6309358744513801, 2.6756891871939668, 2.2462163028632407, 2.834218142935078, 1.3636011413956708, 2.136492677348201, 1.1958477078086263, 0.9812929816286328, 2.111943984124336, 3.362362263424206, 1.3684048508678959, 3.434135850470044, 2.449537479886035, 3.5560013968260376, 1.1307599079427475, 0.682302140012309, 4.225620617348445, 2.1086331270704344, 4.855688442410348, 1.4246667770153378,

In [0]:
"""
Generate semantic embeddings for cleaned job text using SentenceTransformer.
Each cleaned description (summary + skills) is converted to a BERT embedding
and stored as a list of floats in a new column `bert_embedding`.
We do this for train/val/test and save to the Gold layer.
"""

from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, FloatType

# We already have: tfidf_train, tfidf_val, tfidf_test
# (from the previous TF-IDF step)

model_name = "all-MiniLM-L6-v2"

def embed_text(text):
    """
    Given a cleaned text string, return its BERT embedding as a list of floats.
    This function will be wrapped as a Spark UDF to apply on each row in parallel.
    """
    if not text:
        return None
    global model
    try:
        model
    except NameError:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer(model_name)
    return model.encode(text).tolist()

embed_udf = udf(embed_text, ArrayType(FloatType()))

# 1) TRAIN embeddings
embedded_train = tfidf_train.withColumn("bert_embedding", embed_udf("clean_text"))

display(
    embedded_train.select("clean_text", "bert_embedding").limit(5)
)

# 2) VAL embeddings
embedded_val = tfidf_val.withColumn("bert_embedding", embed_udf("clean_text"))

# 3) TEST embeddings
embedded_test = tfidf_test.withColumn("bert_embedding", embed_udf("clean_text"))

# 4) Save to Gold layer (adjust container + account)
base_path = "abfss://lakehouse@projectlinkedinjobs.dfs.core.windows.net/gold/linkedin_feature_v2/"

embedded_train.write.format("delta").mode("overwrite").save(base_path + "embedded_train/")
embedded_val.write.format("delta").mode("overwrite").save(base_path + "embedded_val/")
embedded_test.write.format("delta").mode("overwrite").save(base_path + "embedded_test/")


clean_text bert_embedding description ability to supervise any area of the distribution center or production reviews work load requirements and staffs shift ensuring adequate coverage and within budgeted guidelines ensures productivity and cost controls are met or exceeded creates a climate providing motivation participation and opportunities for employees to do their best prepares staff for future opportunities and takes an active role in the development of future leaders impacts retention by fostering a respectful team environment that supports workplace values through professional communication and timely problem solving responsible for seasonal planning by developing continuous improvement programs and procedures responsible for developing training materials and procedures for department ensures all department procedures sop are documented and utlized ensures system training and appropriate documentation to ensure systems are utilized and users are systems saavy sap collaborate with peer supervisors to promote crossepartmental cooperation and ensure clear communication monitor employee performance and create and deliver pips and pdrs for direct and indirect reports assists in interviewing and assessing leadership candidates as assigned communicates effectively both horizontally and vertically maintaining twoay communications responsible for department and company safety and security programs safety audits and ensuring the safe conduct of all associates within their assigned area coordinate with qa to ensure sqf and fsma requirements are met assist in maintaining the security of the facility requirements education bachelors degree preferred high school diploma required the top candidate for this position will have three to five years of leadership experience in a similar role or level of responsibility time management organizational skills and the ability to influence and motivate a team previous warehouse management system and sap knowledge desired ability to analyze reports and draw conclusionsrecommendations knowledge of microsoft office effective written and verbal communication skills show more show less supervision workload management cost control motivation leadership development team building continuous improvement training and development documentation sap crossdepartmental collaboration performance management interviewing and assessment twoway communication safety and security management sqf and fsma compliance facility security high school diploma bachelors degree preferred leadership experience time management organizational skills influencing and motivating skills warehouse management system microsoft office data analysis written and verbal communication List(-0.054267716, -0.0014250552, -0.020192554, 0.0209525, -0.032126535, 0.025285387, -0.052671667, 0.03361798, -0.025286485, 0.0022865618, 0.013227008, -0.052694608, -0.005484084, 0.013462635, 0.006632999, -0.04095021, 0.027651303, -0.015021416, -0.021154767, -0.1162191, 0.014739296, -0.018659493, -0.045595173, 0.0034493299, -0.1312733, -0.04424418, -0.028728118, -0.030308453, -0.030978557, -0.048088677, 0.05356723, -0.014779153, 0.045703355, 0.032432526, -0.018155843, 0.13237529, 0.060144912, -0.028746841, 0.09556001, 0.037990335, -0.08975325, -0.067410685, -0.062430102, -0.04920724, -0.07141925, -0.057499908, -0.043572363, -0.005244282, -0.027262308, -0.020129735, -0.015353598, -0.011128574, 0.070267595, 0.061364025, 0.005149212, -0.0014119106, 0.06419427, -0.044835206, 7.655052E-4, -0.017101463, -0.082672864, 0.0062938067, -0.033473644, 0.008379995, 0.014952786, -0.05171912, -0.06184861, 0.039057825, 0.024547823, -0.053862695, -0.047497015, -0.05832605, -0.0867154, 0.034255594, 0.009264793, 0.03651181, 0.0398754, 0.05536707, 0.062142804, -0.03706721, 0.075439595, 0.07124108, -0.040237874, 0.08287616, -0.05752908, -0.03309422, -7.227453E-4, 0.03632988, 0.0060132947, 0.017492188, 0.08412298, -0.008779035, -0.032458946, 0.0016724025, -0.02009783, -0.007751966,

In [0]:
"""
Compute readability scores (Flesch Reading Ease) for each cleaned job text
(summary + skills) using textstat. Higher scores = easier to read.

We apply this to train/val/test, add a `readability_score` column,
and (optionally) save the augmented datasets.
"""

import textstat
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

# We already have: embedded_train, embedded_val, embedded_test
# from the previous BERT step

def compute_readability(text):
    """
    Compute Flesch Reading Ease score for a given text.
    Returns 0.0 if the text is empty or invalid.
    """
    if not text:
        return 0.0
    try:
        return float(textstat.flesch_reading_ease(text))
    except:
        return 0.0

readability_udf = udf(compute_readability, DoubleType())

# 1) TRAIN
embedded_train = embedded_train.withColumn(
    "readability_score", readability_udf("clean_text")
)

display(
    embedded_train.select("clean_text", "readability_score").limit(5)
)

# 2) VAL
embedded_val = embedded_val.withColumn(
    "readability_score", readability_udf("clean_text")
)

# 3) TEST
embedded_test = embedded_test.withColumn(
    "readability_score", readability_udf("clean_text")
)

# 4) Optionally save to Gold layer (new paths, or overwrite previous)
base_path = "abfss://lakehouse@projectlinkedinjobs.dfs.core.windows.net/gold/linkedin_feature_v2/"

embedded_train.write.format("delta").mode("overwrite").save(base_path + "embedded_train_readability/")
embedded_val.write.format("delta").mode("overwrite").save(base_path + "embedded_val_readability/")
embedded_test.write.format("delta").mode("overwrite").save(base_path + "embedded_test_readability/")


clean_text readability_score description ability to supervise any area of the distribution center or production reviews work load requirements and staffs shift ensuring adequate coverage and within budgeted guidelines ensures productivity and cost controls are met or exceeded creates a climate providing motivation participation and opportunities for employees to do their best prepares staff for future opportunities and takes an active role in the development of future leaders impacts retention by fostering a respectful team environment that supports workplace values through professional communication and timely problem solving responsible for seasonal planning by developing continuous improvement programs and procedures responsible for developing training materials and procedures for department ensures all department procedures sop are documented and utlized ensures system training and appropriate documentation to ensure systems are utilized and users are systems saavy sap collaborate with peer supervisors to promote crossepartmental cooperation and ensure clear communication monitor employee performance and create and deliver pips and pdrs for direct and indirect reports assists in interviewing and assessing leadership candidates as assigned communicates effectively both horizontally and vertically maintaining twoay communications responsible for department and company safety and security programs safety audits and ensuring the safe conduct of all associates within their assigned area coordinate with qa to ensure sqf and fsma requirements are met assist in maintaining the security of the facility requirements education bachelors degree preferred high school diploma required the top candidate for this position will have three to five years of leadership experience in a similar role or level of responsibility time management organizational skills and the ability to influence and motivate a team previous warehouse management system and sap knowledge desired ability to analyze reports and draw conclusionsrecommendations knowledge of microsoft office effective written and verbal communication skills show more show less supervision workload management cost control motivation leadership development team building continuous improvement training and development documentation sap crossdepartmental collaboration performance management interviewing and assessment twoway communication safety and security management sqf and fsma compliance facility security high school diploma bachelors degree preferred leadership experience time management organizational skills influencing and motivating skills warehouse management system microsoft office data analysis written and verbal communication -339.83771186440674 job summary the primary responsibility of the business development manager is to prospect grow and nurture new business opportunities within the pittsburgh metro area with a focus on generating new revenue and improving profitability in accordance with num krew business objectives this position is tasked with prospecting to identify new potential opportunities and progress them through the sales cycle converting leads into customers the bdm develops and maintains positive relationships with new and existing commercial customers by highlighting industry knowledge and expertise backed by the companys reputation this position supports a fastaced growth environment and regularly communicates with the internal sales and leadership teams this bdm is a salarylusommission position there will also be bonus opportunities based on the overall achievement of the position kpis both the commission scale and kpionus structure are available upon request duties and responsibilities leverages various contact methods such as cold calling or inerson meetings to secure profitable sales through contact lists and selfenerated leads to grow yoy results accomplishes business development by researching and developing marketingales opportunities and plans sells and

In [0]:
"""
Compute subjectivity scores for each cleaned job text using TextBlob.
0 = very objective, 1 = very subjective.

We apply this to train/val/test and add a `subjectivity` column.
"""

from textblob import TextBlob
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

def compute_subjectivity(text):
    """
    Compute subjectivity score using TextBlob.
    Returns 0.0 for empty or invalid text.
    """
    if not text:
        return 0.0
    try:
        return float(TextBlob(text).sentiment.subjectivity)
    except:
        return 0.0

subjectivity_udf = udf(compute_subjectivity, DoubleType())

# TRAIN
embedded_train = embedded_train.withColumn(
    "subjectivity", subjectivity_udf("clean_text")
)

display(
    embedded_train.select("clean_text", "subjectivity").limit(5)
)

# VAL
embedded_val = embedded_val.withColumn(
    "subjectivity", subjectivity_udf("clean_text")
)

# TEST
embedded_test = embedded_test.withColumn(
    "subjectivity", subjectivity_udf("clean_text")
)


clean_text subjectivity description ability to supervise any area of the distribution center or production reviews work load requirements and staffs shift ensuring adequate coverage and within budgeted guidelines ensures productivity and cost controls are met or exceeded creates a climate providing motivation participation and opportunities for employees to do their best prepares staff for future opportunities and takes an active role in the development of future leaders impacts retention by fostering a respectful team environment that supports workplace values through professional communication and timely problem solving responsible for seasonal planning by developing continuous improvement programs and procedures responsible for developing training materials and procedures for department ensures all department procedures sop are documented and utlized ensures system training and appropriate documentation to ensure systems are utilized and users are systems saavy sap collaborate with peer supervisors to promote crossepartmental cooperation and ensure clear communication monitor employee performance and create and deliver pips and pdrs for direct and indirect reports assists in interviewing and assessing leadership candidates as assigned communicates effectively both horizontally and vertically maintaining twoay communications responsible for department and company safety and security programs safety audits and ensuring the safe conduct of all associates within their assigned area coordinate with qa to ensure sqf and fsma requirements are met assist in maintaining the security of the facility requirements education bachelors degree preferred high school diploma required the top candidate for this position will have three to five years of leadership experience in a similar role or level of responsibility time management organizational skills and the ability to influence and motivate a team previous warehouse management system and sap knowledge desired ability to analyze reports and draw conclusionsrecommendations knowledge of microsoft office effective written and verbal communication skills show more show less supervision workload management cost control motivation leadership development team building continuous improvement training and development documentation sap crossdepartmental collaboration performance management interviewing and assessment twoway communication safety and security management sqf and fsma compliance facility security high school diploma bachelors degree preferred leadership experience time management organizational skills influencing and motivating skills warehouse management system microsoft office data analysis written and verbal communication 0.4220833333333333 job summary the primary responsibility of the business development manager is to prospect grow and nurture new business opportunities within the pittsburgh metro area with a focus on generating new revenue and improving profitability in accordance with num krew business objectives this position is tasked with prospecting to identify new potential opportunities and progress them through the sales cycle converting leads into customers the bdm develops and maintains positive relationships with new and existing commercial customers by highlighting industry knowledge and expertise backed by the companys reputation this position supports a fastaced growth environment and regularly communicates with the internal sales and leadership teams this bdm is a salarylusommission position there will also be bonus opportunities based on the overall achievement of the position kpis both the commission scale and kpionus structure are available upon request duties and responsibilities leverages various contact methods such as cold calling or inerson meetings to secure profitable sales through contact lists and selfenerated leads to grow yoy results accomplishes business development by researching and developing marketingales opportunities and plans sells and assis

In [0]:
base_path = "abfss://lakehouse@projectlinkedinjobs.dfs.core.windows.net/gold/linkedin_feature_v2/"

embedded_train.write.format("delta").mode("overwrite").save(base_path + "embedded_train_final/")
embedded_val.write.format("delta").mode("overwrite").save(base_path + "embedded_val_final/")
embedded_test.write.format("delta").mode("overwrite").save(base_path + "embedded_test_final/")


In [0]:
"""
Calculate the average word length for each cleaned job text (summary + skills)
to estimate language complexity. Longer words = more technical/complex language.
"""

import numpy as np
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

def compute_avg_word_length(text):
    """
    Calculate average word length for a given text.
    Returns 0.0 for empty or invalid strings.
    """
    if not text:
        return 0.0
    try:
        words = text.split()
        return float(np.mean([len(w) for w in words])) if words else 0.0
    except:
        return 0.0

avg_word_len_udf = udf(compute_avg_word_length, DoubleType())

# TRAIN
embedded_train = embedded_train.withColumn(
    "avg_word_length", avg_word_len_udf("clean_text")
)

display(
    embedded_train.select("clean_text", "avg_word_length").limit(5)
)

# VAL
embedded_val = embedded_val.withColumn(
    "avg_word_length", avg_word_len_udf("clean_text")
)

# TEST
embedded_test = embedded_test.withColumn(
    "avg_word_length", avg_word_len_udf("clean_text")
)


clean_text avg_word_length description ability to supervise any area of the distribution center or production reviews work load requirements and staffs shift ensuring adequate coverage and within budgeted guidelines ensures productivity and cost controls are met or exceeded creates a climate providing motivation participation and opportunities for employees to do their best prepares staff for future opportunities and takes an active role in the development of future leaders impacts retention by fostering a respectful team environment that supports workplace values through professional communication and timely problem solving responsible for seasonal planning by developing continuous improvement programs and procedures responsible for developing training materials and procedures for department ensures all department procedures sop are documented and utlized ensures system training and appropriate documentation to ensure systems are utilized and users are systems saavy sap collaborate with peer supervisors to promote crossepartmental cooperation and ensure clear communication monitor employee performance and create and deliver pips and pdrs for direct and indirect reports assists in interviewing and assessing leadership candidates as assigned communicates effectively both horizontally and vertically maintaining twoay communications responsible for department and company safety and security programs safety audits and ensuring the safe conduct of all associates within their assigned area coordinate with qa to ensure sqf and fsma requirements are met assist in maintaining the security of the facility requirements education bachelors degree preferred high school diploma required the top candidate for this position will have three to five years of leadership experience in a similar role or level of responsibility time management organizational skills and the ability to influence and motivate a team previous warehouse management system and sap knowledge desired ability to analyze reports and draw conclusionsrecommendations knowledge of microsoft office effective written and verbal communication skills show more show less supervision workload management cost control motivation leadership development team building continuous improvement training and development documentation sap crossdepartmental collaboration performance management interviewing and assessment twoway communication safety and security management sqf and fsma compliance facility security high school diploma bachelors degree preferred leadership experience time management organizational skills influencing and motivating skills warehouse management system microsoft office data analysis written and verbal communication 6.607344632768362 job summary the primary responsibility of the business development manager is to prospect grow and nurture new business opportunities within the pittsburgh metro area with a focus on generating new revenue and improving profitability in accordance with num krew business objectives this position is tasked with prospecting to identify new potential opportunities and progress them through the sales cycle converting leads into customers the bdm develops and maintains positive relationships with new and existing commercial customers by highlighting industry knowledge and expertise backed by the companys reputation this position supports a fastaced growth environment and regularly communicates with the internal sales and leadership teams this bdm is a salarylusommission position there will also be bonus opportunities based on the overall achievement of the position kpis both the commission scale and kpionus structure are available upon request duties and responsibilities leverages various contact methods such as cold calling or inerson meetings to secure profitable sales through contact lists and selfenerated leads to grow yoy results accomplishes business development by researching and developing marketingales opportunities and plans sells and ass

In [0]:
base_path = "abfss://lakehouse@projectlinkedinjobs.dfs.core.windows.net/gold/linkedin_feature_v2/"

embedded_train.write.format("delta").mode("overwrite").save(base_path + "embedded_train_features/")
embedded_val.write.format("delta").mode("overwrite").save(base_path + "embedded_val_features/")
embedded_test.write.format("delta").mode("overwrite").save(base_path + "embedded_test_features/")


In [0]:
"""
Compute the lexical diversity of each cleaned job text (summary + skills) to
measure vocabulary richness. Higher values = more varied language.

We apply this to train/val/test and add a `lexical_diversity` column.
"""

from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

def compute_lexical_diversity(text):
    """
    Compute lexical diversity for a given text.
    Returns 0.0 for empty or invalid text.
    """
    if not text:
        return 0.0
    try:
        words = text.split()
        unique_words = set(words)
        return float(len(unique_words) / len(words)) if words else 0.0
    except:
        return 0.0

lexical_div_udf = udf(compute_lexical_diversity, DoubleType())

# TRAIN
embedded_train = embedded_train.withColumn(
    "lexical_diversity", lexical_div_udf("clean_text")
)

display(
    embedded_train.select("clean_text", "lexical_diversity").limit(5)
)

# VAL
embedded_val = embedded_val.withColumn(
    "lexical_diversity", lexical_div_udf("clean_text")
)

# TEST
embedded_test = embedded_test.withColumn(
    "lexical_diversity", lexical_div_udf("clean_text")
)


clean_text lexical_diversity description ability to supervise any area of the distribution center or production reviews work load requirements and staffs shift ensuring adequate coverage and within budgeted guidelines ensures productivity and cost controls are met or exceeded creates a climate providing motivation participation and opportunities for employees to do their best prepares staff for future opportunities and takes an active role in the development of future leaders impacts retention by fostering a respectful team environment that supports workplace values through professional communication and timely problem solving responsible for seasonal planning by developing continuous improvement programs and procedures responsible for developing training materials and procedures for department ensures all department procedures sop are documented and utlized ensures system training and appropriate documentation to ensure systems are utilized and users are systems saavy sap collaborate with peer supervisors to promote crossepartmental cooperation and ensure clear communication monitor employee performance and create and deliver pips and pdrs for direct and indirect reports assists in interviewing and assessing leadership candidates as assigned communicates effectively both horizontally and vertically maintaining twoay communications responsible for department and company safety and security programs safety audits and ensuring the safe conduct of all associates within their assigned area coordinate with qa to ensure sqf and fsma requirements are met assist in maintaining the security of the facility requirements education bachelors degree preferred high school diploma required the top candidate for this position will have three to five years of leadership experience in a similar role or level of responsibility time management organizational skills and the ability to influence and motivate a team previous warehouse management system and sap knowledge desired ability to analyze reports and draw conclusionsrecommendations knowledge of microsoft office effective written and verbal communication skills show more show less supervision workload management cost control motivation leadership development team building continuous improvement training and development documentation sap crossdepartmental collaboration performance management interviewing and assessment twoway communication safety and security management sqf and fsma compliance facility security high school diploma bachelors degree preferred leadership experience time management organizational skills influencing and motivating skills warehouse management system microsoft office data analysis written and verbal communication 0.5536723163841808 job summary the primary responsibility of the business development manager is to prospect grow and nurture new business opportunities within the pittsburgh metro area with a focus on generating new revenue and improving profitability in accordance with num krew business objectives this position is tasked with prospecting to identify new potential opportunities and progress them through the sales cycle converting leads into customers the bdm develops and maintains positive relationships with new and existing commercial customers by highlighting industry knowledge and expertise backed by the companys reputation this position supports a fastaced growth environment and regularly communicates with the internal sales and leadership teams this bdm is a salarylusommission position there will also be bonus opportunities based on the overall achievement of the position kpis both the commission scale and kpionus structure are available upon request duties and responsibilities leverages various contact methods such as cold calling or inerson meetings to secure profitable sales through contact lists and selfenerated leads to grow yoy results accomplishes business development by researching and developing marketingales opportunities and plans sells and 

In [0]:
base_path = "abfss://lakehouse@projectlinkedinjobs.dfs.core.windows.net/gold/linkedin_feature_v2/"

combined_train = embedded_train
combined_val   = embedded_val
combined_test  = embedded_test

combined_train.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(base_path + "combined_train/")

combined_val.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(base_path + "combined_val/")

combined_test.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(base_path + "combined_test/")


In [0]:
display(combined_train.limit(10))

company search_country job_level job_type job_skills job_summary job_role label raw_text clean_text text_length_words text_length_chars sentiment_pos sentiment_neu sentiment_neg sentiment_compound words filtered_words raw_features tfidf_features bert_embedding readability_score subjectivity avg_word_length lexical_diversity 1-800-flowers.com, inc. united states mid senior onsite supervision, workload management, cost control, motivation, leadership development, team building, continuous improvement, training and development, documentation, sap, crossdepartmental collaboration, performance management, interviewing and assessment, twoway communication, safety and security management, sqf and fsma compliance, facility security, high school diploma, bachelor's degree (preferred), leadership experience, time management, organizational skills, influencing and motivating skills, warehouse management system, microsoft office, data analysis, written and verbal communication description ability to supervise any area of the distribution center or production reviews work load requirements and staffs shift ensuring adequate coverage and within budgeted guidelines. ensures productivity and cost controls are met or exceeded. creates a climate providing motivation, participation and opportunities for employees to do their best. prepares staff for future opportunities and takes an active role in the development of future leaders. impacts retention by fostering a respectful team environment that supports workplace values through professional communication and timely problem solving. responsible for seasonal planning by developing continuous improvement programs and procedures. responsible for developing training materials and procedures for department. ensures all department procedures (sop) are documented and utlized. ensures system training and appropriate documentation to ensure systems are utilized and users are systems saavy (sap) collaborate with peer supervisors to promote cross,epartmental cooperation and ensure clear communication monitor employee performance and create and deliver pip's and pdr's for direct and indirect reports. assists in interviewing and assessing leadership candidates as assigned. communicates effectively both horizontally and vertically maintaining two,ay communications responsible for department and company safety and security programs, safety audits and ensuring the safe conduct of all associates within their assigned area. coordinate with qa to ensure sqf and fsma requirements are met assist in maintaining the security of the facility requirements education , bachelor's degree preferred. high school diploma required. the top candidate for this position will have three to five years of leadership experience in a similar role or level of responsibility. time management, organizational skills and the ability to influence and motivate a team previous warehouse management system and sap knowledge desired. ability to analyze reports and draw conclusions,recommendations knowledge of microsoft office effective written and verbal communication skills show more show less supervisor 77.0 description ability to supervise any area of the distribution center or production reviews work load requirements and staffs shift ensuring adequate coverage and within budgeted guidelines. ensures productivity and cost controls are met or exceeded. creates a climate providing motivation, participation and opportunities for employees to do their best. prepares staff for future opportunities and takes an active role in the development of future leaders. impacts retention by fostering a respectful team environment that supports workplace values through professional communication and timely problem solving. responsible for seasonal planning by developing continuous improvement programs and procedures. responsible for developing training materials and procedures for department. ensures all department procedures (sop) are documented and utlize